In [1]:
import sys

sys.executable, sys.version

('/usr/bin/python3', '3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]')

In [2]:
# %load_ext autoreload
# %autoreload 2

## Install packages

In [3]:
!uv pip install \
    'transformers>=4.57.3,<5.0.0' \
    'torch>=2.8.0,<2.10.0' \
    'torchvision>=0.23.0,<0.25.0' \
    timm \
    pillow \
    loguru \
    gdown

Using Python 3.12.12 environment at: /usr
Resolved 49 packages in 308ms                                        
Prepared 7 packages in 14.79s                                            
Uninstalled 6 packages in 947ms
Installed 7 packages in 228ms                               
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 + loguru==0.7.3
 - nvidia-nvshmem-cu12==3.4.5
 + nvidia-nvshmem-cu12==3.3.20
 - torch==2.10.0+cu128
 + torch==2.9.1
 - torchvision==0.25.0+cu128
 + torchvision==0.24.1
 - transformers==5.0.0
 + transformers==4.57.6
 - triton==3.6.0
 + triton==3.5.1


## Import packages

In [4]:
import json
import os
from collections import defaultdict
from typing import Any

import numpy as np
import torch
import transformers
from loguru import logger
from PIL import ImageFile
from pydantic import BaseModel
from torch import Tensor
from transformers import AutoModel

ImageFile.LOAD_TRUNCATED_IMAGES = True

transformers.__version__, torch.__version__

('4.57.6', '2.9.1+cu128')

## Loading model

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = AutoModel.from_pretrained("minhnguyent546/ViSigLIP-OT", trust_remote_code=True)
model.to(device)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/767 [00:00<?, ?B/s]

configuration_viclip_ot.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/minhnguyent546/ViSigLIP-OT:
- configuration_viclip_ot.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

ViCLIPOTModel(
  (text_model): ViCLIPOTTextModel(
    (encoder): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(64001, 768, padding_idx=1)
        (position_embeddings): Embedding(258, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense): Linear

## Preparing dataset

In [6]:
!gdown --fuzzy https://drive.google.com/file/d/1mQ-TUv_l3pDPfCIsZq0YXUtsem68HzM5/view?usp=drive_link

Downloading...
From (original): https://drive.google.com/uc?id=1mQ-TUv_l3pDPfCIsZq0YXUtsem68HzM5
From (redirected): https://drive.google.com/uc?id=1mQ-TUv_l3pDPfCIsZq0YXUtsem68HzM5&confirm=t&uuid=209f5b3d-e195-4e19-bb85-4344034db6fb
To: /content/UIT-OpenViIC.tar.gz
100% 3.81G/3.81G [00:50<00:00, 75.8MB/s]


In [7]:
!tar -xzf UIT-OpenViIC.tar.gz

In [8]:
DATASET_DIR = "./UIT-OpenViIC"
TEST_SPLIT_FILE = "test.json"
BATCH_SIZE = 64

In [9]:
class ImageTextDataImage(BaseModel):
    id: int | str
    image_path: str


class ImageTextDataAnnotation(BaseModel):
    id: int | str
    caption: str
    image_id: int | str


class ImageTextData(BaseModel):
    images: list[ImageTextDataImage]
    annotations: list[ImageTextDataAnnotation]


metadata_file_path = os.path.join(DATASET_DIR, TEST_SPLIT_FILE)

logger.info(f"Loading image text data from: {metadata_file_path}")
with open(metadata_file_path, "r") as f:
    metadata = ImageTextData.model_validate(json.load(f))

logger.info(f"Found {len(metadata.images)} images and {len(metadata.annotations)} annotations.")
id_to_image_path = {image.id: image.image_path for image in metadata.images}

captions_by_image_id: dict[int | str, list[tuple[int | str, str]]] = defaultdict(list)
for annotation in metadata.annotations:
    image_id = annotation.image_id
    if image_id not in id_to_image_path:
        raise RuntimeError(
            f"Could not find image with ID {image_id} for annotation {annotation.id}"
        )

    captions_by_image_id[image_id].append((annotation.id, annotation.caption))

# test_samples: (imag_id, image_path, list of captions)
test_samples: list[tuple[int | str, str, list[str]]] = []
pair_count = 0
for image_id in sorted(captions_by_image_id.keys()):
    captions_by_image_id[image_id].sort(key=lambda x: x[0])  # sort by caption_id
    captions = [caption for _caption_id, caption in captions_by_image_id[image_id]]
    image_path = os.path.join(DATASET_DIR, id_to_image_path[image_id])
    test_samples.append((image_id, image_path, captions))

2026-02-23 01:04:24.205 | INFO     | __main__:<cell line: 0>:19 - Loading image text data from: ./UIT-OpenViIC/test.json
2026-02-23 01:04:24.245 | INFO     | __main__:<cell line: 0>:23 - Found 2001 images and 10001 annotations.


In [10]:
image_ids = [sample[0] for sample in test_samples]
image_paths = [sample[1] for sample in test_samples]
captions = [caption for sample in test_samples for caption in sample[2]]
num_captions_per_image = [len(sample[2]) for sample in test_samples]

In [11]:
image_ids[:5], image_paths[:5], captions[:10], num_captions_per_image[:5]

([245, 3989, 5194, 11742, 12409],
 ['./UIT-OpenViIC/./images/00000007772.jpg',
  './UIT-OpenViIC/./images/00000005045.jpg',
  './UIT-OpenViIC/./images/00000009679.jpg',
  './UIT-OpenViIC/./images/00000009683.jpg',
  './UIT-OpenViIC/./images/00000005138.jpg'],
 ['có một đứa bé mặc áo khoác hồng ngồi kế người phụ nữ mặc đồ đỏ đang đứng',
  'người đàn ông mặc áo sọc đang đưa tay ra lấy đồ phía sau chiếc xe đẩy',
  'những người phụ nữ đang ngồi bên cạnh xe bán thức ăn',
  'người phụ nữ mang bộ áo quần màu đỏ đang đứng cạnh một bé gái',
  'người đàn ông mặc áo xọc đang đứng cạnh chiếc tủ và chiếc bàn đồ ăn',
  'nhiều người đang đứng thành hàng xem múa lân',
  'hai chú lân màu vàng đang đứng trước tấm bảng màu cam',
  'nhiều người đang đứng xe hai con lân nhảy trước một cửa hàng khai trương',
  'trước mặt cửa hàng có hai con lân màu cam đang nhảy',
  'rất nhiều người đứng trước cửa hàng xem múa lân'],
 [5, 5, 5, 5, 5])

## Computing embeddings

In [12]:
text_embeddings = model.encode_text(
    sentences=captions,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize=True,
    padding=True,
    truncation=True,
    max_length=512,
)

Encoding sentences:   0%|          | 0/157 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/422 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [13]:
image_embeddings = model.encode_image(
    images=image_paths,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize=True,
)

Encoding images:   0%|          | 0/32 [00:00<?, ?it/s]

In [14]:
text_embeddings = text_embeddings.cpu()
image_embeddings = image_embeddings.cpu()

# repeat image embeddings to match number of captions for each image
image_embeddings = image_embeddings.repeat_interleave(
    torch.tensor(
        num_captions_per_image,
        device=image_embeddings.device,
    ),
    dim=0,
)

image_ids = torch.tensor(image_ids).repeat_interleave(
    torch.tensor(num_captions_per_image),
    dim=0,
)

In [15]:
text_embeddings.shape, image_embeddings.shape, image_ids.shape

(torch.Size([10001, 768]), torch.Size([10001, 768]), torch.Size([10001]))

In [16]:
def compute_retrieval_metrics(
    logits: Tensor, mask: Tensor, prefix: str, k_vals=(1, 5, 10)
) -> dict[str, Any]:
    """ "Compute recall@k and mean rank."""

    results = {}
    max_k = min(max(k_vals), logits.shape[1])
    _, top_indices = logits.topk(max_k, dim=1)  # [B, max_k]

    # gather ground truth booleans at the retrieved positions
    rows = torch.arange(logits.shape[0]).view(-1, 1)
    retrieved_mask = mask[rows, top_indices]  # [B, max_k]

    for k in k_vals:
        # hit if at least one of the top k is True
        hits = retrieved_mask[:, :k].any(dim=1)
        results[f"{prefix}_R__{k}"] = hits.float().mean().item()

    argsort = torch.argsort(logits, dim=1, descending=True)

    # sorted_mask[i, j] is True if the item at rank 'j' is a match
    sorted_mask = mask[rows, argsort]

    # find the first rank (min index) where sorted_mask is True
    rank_matrix = torch.arange(logits.shape[1]).view(1, -1).float()
    masked_ranks = rank_matrix.expand(logits.shape[0], -1).clone()
    masked_ranks[~sorted_mask] = float("inf")

    # get the "Best Rank" (lowest index) for every row
    best_rank_per_row = masked_ranks.min(dim=1).values

    # convert 0-indexed to 1-indexed
    best_rank_per_row = best_rank_per_row.numpy() + 1

    results[f"{prefix}_mean_rank"] = np.mean(best_rank_per_row)
    results[f"{prefix}_median_rank"] = np.floor(np.median(best_rank_per_row))

    return results


def get_retrieval_metrics(
    image_features: Tensor,
    text_features: Tensor,
    image_ids: Tensor | None = None,
) -> dict[str, Any]:
    """
    If `image_ids` is provided, the computation will take into account image with multiple captions.
    """
    metrics: dict[str, Any] = {}
    if image_ids is None:
        # 1-1 image caption mapping
        image_ids = torch.arange(len(image_features))

    image_features = image_features.cpu().float()
    text_features = text_features.cpu().float()
    image_ids = image_ids.cpu()
    image_ids = image_ids.cpu()

    unique_ids, first_indices = np.unique(image_ids.numpy(), return_index=True)
    unique_ids = torch.from_numpy(unique_ids)
    first_indices = torch.from_numpy(first_indices)

    unique_image_features = image_features[first_indices]

    # Image-to-Text
    # Query:   Unique Images [N_unique]
    # Gallery: All Texts     [N_total]
    # Protocol: For each unique image, did we find ANY of its captions?
    # Logits: [N_unique, N_total]
    logits_i2t = unique_image_features @ text_features.t()

    # Mask: [N_unique, N_total]
    # Rows are Unique IDs, Cols are All IDs. Match if they are equal.
    mask_i2t = unique_ids.view(-1, 1) == image_ids.view(1, -1)

    metrics.update(compute_retrieval_metrics(logits_i2t, mask_i2t, prefix="i2t"))

    # Text-to-Image
    # Query:   All Texts     [N_total]
    # Gallery: Unique Images [N_unique]
    # Protocol: For each caption, did we find the ONE correct image?
    # Logits: [N_total, N_unique]
    logits_t2i = text_features @ unique_image_features.t()

    # Mask: [N_total, N_unique]
    # Rows are All IDs, Cols are Unique IDs. Match if they are equal.
    mask_t2i = image_ids.view(-1, 1) == unique_ids.view(1, -1)

    metrics.update(compute_retrieval_metrics(logits_t2i, mask_t2i, prefix="t2i"))

    return metrics

In [17]:
metrics = get_retrieval_metrics(
    image_features=image_embeddings, text_features=text_embeddings, image_ids=image_ids
)

In [18]:
metrics

{'i2t_R__1': 0.572213888168335,
 'i2t_R__5': 0.8390804529190063,
 'i2t_R__10': 0.9080459475517273,
 'i2t_mean_rank': np.float32(5.8135934),
 'i2t_median_rank': np.float32(1.0),
 't2i_R__1': 0.3914608657360077,
 't2i_R__5': 0.6661334037780762,
 't2i_R__10': 0.7600240111351013,
 't2i_mean_rank': np.float32(25.322968),
 't2i_median_rank': np.float32(2.0)}